# Feature Engineering Dashboard

This notebook creates visualizations for feature engineering slides:

1. Load and merge sensor + glucose data
2. Apply feature engineering pipeline
3. Visualize feature distributions and relationships
4. Analyze feature importance and correlations
5. Export high-quality plots for presentation slides


## 1. Imports and Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import feature engineering functions
import feature_engineering as fe
import model as mdl

# Set style for publication-quality plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

# Set DPI for high-quality output
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

print("Imports successful")


## 2. Load and Prepare Data


In [ ]:
# Load and merge data
base_path = "."
print("Loading data...")
raw_df = mdl.load_and_patch_data(base_path)

print(f"\nRaw data shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns[:10])}...")
print(f"Date range: {raw_df['DATE'].min()} to {raw_df['DATE'].max()}")
print(f"Glucose range: {raw_df['glucose'].min():.1f} - {raw_df['glucose'].max():.1f} mg/dL")

raw_df.head()


## 3. Apply Feature Engineering


In [ ]:
# Apply feature engineering
print("Applying feature engineering pipeline...")
features_df = fe.create_advanced_features(raw_df)

# Filter valid glucose values
features_df = features_df.dropna(subset = ['glucose'])
features_df = features_df[(features_df['glucose'] >= 30) & (features_df['glucose'] <= 500)]

print(f"\nAfter feature engineering:")
print(f"Shape: {features_df.shape}")
print(f"Features created: {features_df.shape[1] - raw_df.shape[1]} new features")

# Get feature columns (exclude metadata)
numeric_df = features_df.select_dtypes(include = [np.number])
metadata = ['glucose', 'chunk_id', 'hours_elapsed', 'session_id', 'DATE']
feature_cols = [c for c in numeric_df.columns if c not in metadata]

print(f"Total feature columns: {len(feature_cols)}")
features_df.head()


## 4. Visualizations


### 4.1 Feature Engineering Pipeline Overview


In [ ]:
# Compare raw vs engineered features
fig, axes = plt.subplots(1, 2, figsize = (14, 6))

# Left: Feature count comparison
categories = ['Raw Features', 'Engineered Features']
counts = [len([c for c in raw_df.columns if c not in ['DATE', 'source_file']]), len(feature_cols)]
colors = ['#2E86AB', '#A23B72']

axes[0].bar(categories, counts, color = colors, alpha = 0.8, edgecolor = 'black', linewidth = 1.5)
axes[0].set_ylabel('Number of Features', fontsize = 12, fontweight = 'bold')
axes[0].set_title('Feature Count: Before vs After Engineering', fontsize = 13, fontweight = 'bold')
axes[0].grid(True, alpha = 0.3, axis = 'y')
for i, (cat, count) in enumerate(zip(categories, counts)):
    axes[0].text(i, count + max(counts)*0.02, str(count), ha = 'center', va = 'bottom', 
                 fontsize = 12, fontweight = 'bold')

# Right: Feature categories
feature_categories = {
    'Temperature': len([c for c in feature_cols if 'temp' in c.lower() or 'T' in c]),
    'N0 Signal': len([c for c in feature_cols if 'N0' in c]),
    'Gradients': len([c for c in feature_cols if 'grad' in c.lower()]),
    'Temporal': len([c for c in feature_cols if any(x in c.lower() for x in ['hour', 'day', 'time', 'rolling', 'ema'])]),
    'Interactions': len([c for c in feature_cols if 'interaction' in c.lower() or 'ratio' in c.lower()]),
    'Other': len([c for c in feature_cols if not any(x in c.lower() for x in ['temp', 'n0', 'grad', 'hour', 'day', 'time', 'rolling', 'ema', 'interaction', 'ratio'])])
}

cat_names = list(feature_categories.keys())
cat_counts = list(feature_categories.values())

axes[1].barh(cat_names, cat_counts, color = plt.cm.Set3(range(len(cat_names))), 
            alpha = 0.8, edgecolor = 'black', linewidth = 1)
axes[1].set_xlabel('Number of Features', fontsize = 12, fontweight = 'bold')
axes[1].set_title('Feature Categories', fontsize = 13, fontweight = 'bold')
axes[1].grid(True, alpha = 0.3, axis = 'x')
for i, (name, count) in enumerate(zip(cat_names, cat_counts)):
    axes[1].text(count + max(cat_counts)*0.02, i, str(count), ha = 'left', va = 'center', 
                fontsize = 11, fontweight = 'bold')

plt.tight_layout()
plt.savefig('feature_engineering_overview.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: feature_engineering_overview.png")


### 4.2 Feature Correlation with Glucose


In [ ]:
# Calculate correlations
X = numeric_df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
y = numeric_df['glucose']

corr_df = fe.analyze_feature_correlations(X, y, feature_cols)

# Get top 25 features by correlation
top_n = 25
top_features = corr_df.head(top_n)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize = (16, 8))

# Left: Top features by Pearson correlation
ax1 = axes[0]
top_pearson = top_features.sort_values('abs_pearson', ascending = True).tail(15)
feature_names_clean = [name.replace('_', ' ').title() for name in top_pearson.index]
# Abbreviate long names
feature_names_clean = [name.replace('Normalized', 'Norm').replace('Rolling', 'Roll') 
                      for name in feature_names_clean]

y_pos = np.arange(len(feature_names_clean))
bars1 = ax1.barh(y_pos, top_pearson['abs_pearson'].values, color = 'steelblue', alpha = 0.8)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(feature_names_clean, fontsize = 9)
ax1.set_xlabel('Absolute Pearson Correlation', fontsize = 12, fontweight = 'bold')
ax1.set_title(f'Top 15 Features: Correlation with Glucose', fontsize = 13, fontweight = 'bold')
ax1.grid(True, alpha = 0.3, axis = 'x')
# Add value labels
for i, (bar, val) in enumerate(zip(bars1, top_pearson['abs_pearson'].values)):
    ax1.text(val + 0.01, i, f'{val:.3f}', va = 'center', fontsize = 8)

# Right: Correlation distribution
ax2 = axes[1]
all_corrs = corr_df['abs_pearson'].values
ax2.hist(all_corrs, bins = 30, color = 'coral', alpha = 0.7, edgecolor = 'black', linewidth = 1)
mean_corr = np.mean(all_corrs)
median_corr = np.median(all_corrs)
ax2.axvline(mean_corr, color = 'red', linestyle = '--', linewidth = 2, 
           label = f'Mean: {mean_corr:.3f}')
ax2.axvline(median_corr, color = 'blue', linestyle = '--', linewidth = 2, 
           label = f'Median: {median_corr:.3f}')
ax2.set_xlabel('Absolute Pearson Correlation', fontsize = 12, fontweight = 'bold')
ax2.set_ylabel('Number of Features', fontsize = 12, fontweight = 'bold')
ax2.set_title('Distribution of Feature Correlations', fontsize = 13, fontweight = 'bold')
ax2.legend()
ax2.grid(True, alpha = 0.3, axis = 'y')

plt.tight_layout()
plt.savefig('feature_correlations.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: feature_correlations.png")


In [ ]:
# Perform feature selection analysis
from sklearn.feature_selection import SelectKBest, f_regression

X_clean = X.fillna(0).replace([np.inf, -np.inf], 0)
y_clean = y.fillna(y.mean())

selector = SelectKBest(score_func = f_regression, k = 'all')
selector.fit(X_clean, y_clean)

f_scores = selector.scores_
f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-10)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'f_score': f_scores,
    'f_score_norm': f_norm,
    'abs_pearson_corr': corr_df.loc[feature_cols, 'abs_pearson']
}).sort_values('f_score', ascending = False)

# Get top 20 features
top_20 = importance_df.head(20)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize = (16, 8))

# Left: F-scores
ax1 = axes[0]
top_f = top_20.sort_values('f_score', ascending = True)
feature_names_clean = [name.replace('_', ' ').title() for name in top_f['feature']]
feature_names_clean = [name.replace('Normalized', 'Norm').replace('Rolling', 'Roll')
                      .replace('Gradient', 'Grad').replace('Percentage', 'Pct') 
                      for name in feature_names_clean]

y_pos = np.arange(len(feature_names_clean))
bars1 = ax1.barh(y_pos, top_f['f_score'].values, color = '#2E86AB', alpha = 0.8)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(feature_names_clean, fontsize = 9)
ax1.set_xlabel('F-Score (Statistical Significance)', fontsize = 12, fontweight = 'bold')
ax1.set_title('Top 20 Features: F-Score Importance', fontsize = 13, fontweight = 'bold')
ax1.grid(True, alpha = 0.3, axis = 'x')

# Right: Combined importance (normalized F-score + correlation)
ax2 = axes[1]
combined_score = (top_20['f_score_norm'] * 0.6 + top_20['abs_pearson_corr'] * 0.4)
top_combined = top_20.copy()
top_combined['combined'] = combined_score
top_combined = top_combined.sort_values('combined', ascending = True)

bars2 = ax2.barh(y_pos, top_combined['combined'].values, color = '#A23B72', alpha = 0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(feature_names_clean, fontsize = 9)
ax2.set_xlabel('Combined Importance Score', fontsize = 12, fontweight = 'bold')
ax2.set_title('Top 20 Features: Combined Importance\n(60% F-Score + 40% Correlation)', 
             fontsize = 13, fontweight = 'bold')
ax2.grid(True, alpha = 0.3, axis = 'x')

plt.tight_layout()
plt.savefig('feature_importance_analysis.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: feature_importance_analysis.png")


### 4.4 Key Feature Examples: Before vs After Engineering


In [ ]:
# Compare raw N0 signal vs engineered features
if 'N0' in raw_df.columns and 'N0_normalized_30' in features_df.columns:
    fig, axes = plt.subplots(2, 2, figsize = (16, 10))
    
    # Sample data for visualization
    sample_size = min(2000, len(features_df))
    sample_idx = np.random.choice(len(features_df), sample_size, replace = False)
    sample_df = features_df.iloc[sample_idx].sort_values('DATE')
    
    # Top left: Raw N0 signal
    ax1 = axes[0, 0]
    if 'N0' in raw_df.columns:
        raw_sample = raw_df.iloc[sample_idx].sort_values('DATE')
        ax1.plot(raw_sample['DATE'].values[:500], raw_sample['N0'].values[:500], 
                color = 'blue', alpha = 0.6, linewidth = 1.5, label = 'Raw N0')
    ax1.set_xlabel('Time', fontsize = 11)
    ax1.set_ylabel('N0 Signal', fontsize = 11)
    ax1.set_title('Raw N0 Signal (Subject to Drift)', fontsize = 12, fontweight = 'bold')
    ax1.legend()
    ax1.grid(True, alpha = 0.3)
    
    # Top right: Normalized N0 (drift-robust)
    ax2 = axes[0, 1]
    ax2.plot(sample_df['DATE'].values[:500], sample_df['N0_normalized_30'].values[:500], 
            color = 'green', alpha = 0.7, linewidth = 1.5, label = 'Normalized N0 (30-window)')
    ax2.axhline(y = 0, color = 'red', linestyle = '--', linewidth = 1, alpha = 0.5)
    ax2.set_xlabel('Time', fontsize = 11)
    ax2.set_ylabel('Normalized N0 (Z-score)', fontsize = 11)
    ax2.set_title('Drift-Robust: Rolling Normalized N0', fontsize = 12, fontweight = 'bold')
    ax2.legend()
    ax2.grid(True, alpha = 0.3)
    
    # Bottom left: Temperature features
    ax3 = axes[1, 0]
    ax3_twin = None
    if 'temp_mean' in features_df.columns:
        ax3.plot(sample_df['DATE'].values[:500], sample_df['temp_mean'].values[:500], 
                color = 'orange', alpha = 0.7, linewidth = 1.5, label = 'Mean Temperature')
    if 'temp_range' in features_df.columns:
        ax3_twin = ax3.twinx()
        ax3_twin.plot(sample_df['DATE'].values[:500], sample_df['temp_range'].values[:500], 
                     color = 'purple', alpha = 0.7, linewidth = 1.5, label = 'Temperature Range')
        ax3_twin.set_ylabel('Temperature Range (°C)', fontsize = 11, color = 'purple')
        ax3_twin.tick_params(axis = 'y', labelcolor = 'purple')
    ax3.set_xlabel('Time', fontsize = 11)
    ax3.set_ylabel('Mean Temperature (°C)', fontsize = 11, color = 'orange')
    ax3.tick_params(axis = 'y', labelcolor = 'orange')
    ax3.set_title('Temperature Features', fontsize = 12, fontweight = 'bold')
    ax3.legend(loc = 'upper left')
    if ax3_twin is not None:
        ax3_twin.legend(loc = 'upper right')
    ax3.grid(True, alpha = 0.3)
    
    # Bottom right: N0 gradient features
    ax4 = axes[1, 1]
    ax4_twin = None
    if 'N0_pct_change' in features_df.columns:
        ax4.plot(sample_df['DATE'].values[:500], sample_df['N0_pct_change'].values[:500], 
                color = 'red', alpha = 0.7, linewidth = 1.5, label = 'N0 % Change')
    if 'N0_diff' in features_df.columns:
        ax4_twin = ax4.twinx()
        ax4_twin.plot(sample_df['DATE'].values[:500], sample_df['N0_diff'].values[:500], 
                     color = 'teal', alpha = 0.7, linewidth = 1.5, label = 'N0 Difference')
        ax4_twin.set_ylabel('N0 Difference', fontsize = 11, color = 'teal')
        ax4_twin.tick_params(axis = 'y', labelcolor = 'teal')
        ax4_twin.axhline(y = 0, color = 'black', linestyle = ':', linewidth = 1, alpha = 0.5)
    ax4.set_xlabel('Time', fontsize = 11)
    ax4.set_ylabel('N0 % Change', fontsize = 11, color = 'red')
    ax4.tick_params(axis = 'y', labelcolor = 'red')
    ax4.set_title('N0 Gradient Features (Rate of Change)', fontsize = 12, fontweight = 'bold')
    ax4.legend(loc = 'upper left')
    if ax4_twin is not None:
        ax4_twin.legend(loc = 'upper right')
    ax4.grid(True, alpha = 0.3)
    
    plt.tight_layout()
    plt.savefig('feature_examples.png', dpi = 300, bbox_inches = 'tight')
    plt.show()
    print("Saved: feature_examples.png")


### 4.5 Feature Statistics Summary


In [ ]:
# Create summary statistics table visualization
feature_stats = pd.DataFrame({
    'Feature': feature_cols,
    'Mean': [X[c].mean() for c in feature_cols],
    'Std': [X[c].std() for c in feature_cols],
    'Min': [X[c].min() for c in feature_cols],
    'Max': [X[c].max() for c in feature_cols],
    'Correlation': corr_df.loc[feature_cols, 'abs_pearson'].values
})

# Get top features by category
top_by_category = {
    'Temperature': feature_stats[feature_stats['Feature'].str.contains('temp', case = False, na = False)].nlargest(5, 'Correlation'),
    'N0 Signal': feature_stats[feature_stats['Feature'].str.contains('N0', na = False)].nlargest(5, 'Correlation'),
    'Gradients': feature_stats[feature_stats['Feature'].str.contains('grad', case = False, na = False)].nlargest(5, 'Correlation'),
    'Temporal': feature_stats[feature_stats['Feature'].str.contains('rolling|ema|hour|day', case = False, na = False)].nlargest(5, 'Correlation')
}

# Create visualization
fig, ax = plt.subplots(figsize = (14, 8))

categories = list(top_by_category.keys())
y_pos_base = np.arange(len(categories))
bar_width = 0.15
colors_list = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E']

for i, (cat, df) in enumerate(top_by_category.items()):
    top_5 = df.head(5)
    correlations = top_5['Correlation'].values
    x_pos = y_pos_base[i] + np.linspace(-bar_width*2, bar_width*2, len(correlations))
    
    for j, (x, corr, feat) in enumerate(zip(x_pos, correlations, top_5['Feature'])):
        ax.barh(x, corr, height = bar_width*0.8, color = colors_list[j], alpha = 0.8, 
               edgecolor = 'black', linewidth = 0.5)
        # Add feature name (abbreviated)
        feat_short = feat.replace('_', ' ').replace('Normalized', 'Norm')[:25]
        ax.text(corr + 0.01, x, feat_short, va = 'center', fontsize = 7)

ax.set_yticks(y_pos_base)
ax.set_yticklabels(categories, fontsize = 12, fontweight = 'bold')
ax.set_xlabel('Absolute Pearson Correlation with Glucose', fontsize = 12, fontweight = 'bold')
ax.set_title('Top Features by Category: Correlation with Glucose', fontsize = 14, fontweight = 'bold')
ax.grid(True, alpha = 0.3, axis = 'x')
ax.set_xlim([0, max([df['Correlation'].max() for df in top_by_category.values()]) * 1.2])

plt.tight_layout()
plt.savefig('feature_statistics.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: feature_statistics.png")


## 6. Additional Visualizations


### 6.1 Signal Drift: Session A vs Session B


In [ ]:
# Identify sessions based on source files
if 'source_file' in raw_df.columns:
    # Session A: Sensor Data 1_2, Session B: Sensor Data 2_2
    session_a = raw_df[raw_df['source_file'].str.contains('Sensor Data 1', na = False)].copy()
    session_b = raw_df[raw_df['source_file'].str.contains('Sensor Data 2', na = False)].copy()
else:
    # Fallback: split by date or use first/last half
    mid_point = len(raw_df) // 2
    session_a = raw_df.iloc[:mid_point].copy()
    session_b = raw_df.iloc[mid_point:].copy()
    session_a['source_file'] = 'Session A'
    session_b['source_file'] = 'Session B'

# Apply feature engineering to both sessions
print("Processing Session A...")
session_a_feat = fe.create_advanced_features(session_a)
print("Processing Session B...")
session_b_feat = fe.create_advanced_features(session_b)

# Visualize signal drift
if 'N0' in session_a_feat.columns and 'N0' in session_b_feat.columns:
    fig, axes = plt.subplots(2, 2, figsize = (16, 10))
    
    # Sample data for visualization
    sample_a = session_a_feat.sample(min(3000, len(session_a_feat)), random_state = 42).sort_values('DATE')
    sample_b = session_b_feat.sample(min(3000, len(session_b_feat)), random_state = 42).sort_values('DATE')
    
    # Top left: Raw N0 signal comparison
    ax1 = axes[0, 0]
    ax1.plot(sample_a['DATE'].values[:1000], sample_a['N0'].values[:1000], 
            color = 'blue', alpha = 0.6, linewidth = 1.5, label = 'Session A (Raw N0)')
    ax1.plot(sample_b['DATE'].values[:1000], sample_b['N0'].values[:1000], 
            color = 'red', alpha = 0.6, linewidth = 1.5, label = 'Session B (Raw N0)')
    ax1.set_xlabel('Time', fontsize = 11)
    ax1.set_ylabel('N0 Signal', fontsize = 11)
    ax1.set_title('Raw N0 Signal: Session A vs Session B\n(Shows Baseline Drift)', 
                  fontsize = 12, fontweight = 'bold')
    ax1.legend()
    ax1.grid(True, alpha = 0.3)
    
    # Top right: Normalized N0 (drift-corrected)
    ax2 = axes[0, 1]
    if 'N0_normalized_30' in session_a_feat.columns:
        ax2.plot(sample_a['DATE'].values[:1000], sample_a['N0_normalized_30'].values[:1000], 
                color = 'blue', alpha = 0.7, linewidth = 1.5, label = 'Session A (Normalized)')
    if 'N0_normalized_30' in session_b_feat.columns:
        ax2.plot(sample_b['DATE'].values[:1000], sample_b['N0_normalized_30'].values[:1000], 
                color = 'red', alpha = 0.7, linewidth = 1.5, label = 'Session B (Normalized)')
    ax2.axhline(y = 0, color = 'black', linestyle = '--', linewidth = 1, alpha = 0.5)
    ax2.set_xlabel('Time', fontsize = 11)
    ax2.set_ylabel('Normalized N0 (Z-score)', fontsize = 11)
    ax2.set_title('Drift-Corrected: Normalized N0\n(Aligned Baselines)', 
                  fontsize = 12, fontweight = 'bold')
    ax2.legend()
    ax2.grid(True, alpha = 0.3)
    
    # Bottom left: N0 distribution comparison
    ax3 = axes[1, 0]
    ax3.hist(session_a_feat['N0'].dropna().values, bins = 50, alpha = 0.6, 
            color = 'blue', label = 'Session A', edgecolor = 'black', linewidth = 0.5)
    ax3.hist(session_b_feat['N0'].dropna().values, bins = 50, alpha = 0.6, 
            color = 'red', label = 'Session B', edgecolor = 'black', linewidth = 0.5)
    ax3.set_xlabel('N0 Signal Value', fontsize = 11)
    ax3.set_ylabel('Frequency', fontsize = 11)
    ax3.set_title('N0 Distribution: Session A vs Session B', fontsize = 12, fontweight = 'bold')
    ax3.legend()
    ax3.grid(True, alpha = 0.3, axis = 'y')
    
    # Bottom right: Mean N0 over time (showing drift)
    ax4 = axes[1, 1]
    if len(session_a_feat) > 0:
        session_a_feat['hour'] = (session_a_feat['DATE'] - session_a_feat['DATE'].min()).dt.total_seconds() / 3600
        session_a_hourly = session_a_feat.groupby(session_a_feat['hour'] // 1)['N0'].mean()
        ax4.plot(session_a_hourly.index, session_a_hourly.values, 
                color = 'blue', alpha = 0.7, linewidth = 2, marker = 'o', markersize = 3, label = 'Session A Mean')
    if len(session_b_feat) > 0:
        session_b_feat['hour'] = (session_b_feat['DATE'] - session_b_feat['DATE'].min()).dt.total_seconds() / 3600
        session_b_hourly = session_b_feat.groupby(session_b_feat['hour'] // 1)['N0'].mean()
        ax4.plot(session_b_hourly.index, session_b_hourly.values, 
                color = 'red', alpha = 0.7, linewidth = 2, marker = 's', markersize = 3, label = 'Session B Mean')
    ax4.set_xlabel('Hours Elapsed', fontsize = 11)
    ax4.set_ylabel('Mean N0 Signal', fontsize = 11)
    ax4.set_title('Temporal Drift: Mean N0 Over Time', fontsize = 12, fontweight = 'bold')
    ax4.legend()
    ax4.grid(True, alpha = 0.3)
    
    plt.tight_layout()
    plt.savefig('signal_drift_sessions.png', dpi = 300, bbox_inches = 'tight')
    plt.show()
    print("Saved: signal_drift_sessions.png")


### 6.2 1-Hour Chunk Splitting Visualization


In [ ]:
# Apply chunk splitting
chunked_df = mdl.create_time_chunks(features_df, chunk_hours = 1)

# Visualize chunk splitting
fig, axes = plt.subplots(2, 1, figsize = (16, 10))

# Top: Chunk IDs over time
ax1 = axes[0]
sample_size = min(5000, len(chunked_df))
sample_df = chunked_df.sample(sample_size, random_state = 42).sort_values('DATE')

ax1.scatter(sample_df['DATE'].values, sample_df['chunk_id'].values, 
           c = sample_df['chunk_id'].values, cmap = 'tab20', alpha = 0.6, s = 10)
ax1.set_xlabel('Date/Time', fontsize = 11)
ax1.set_ylabel('Chunk ID (1-hour intervals)', fontsize = 11)
ax1.set_title('1-Hour Chunk Splitting: Temporal Segmentation', fontsize = 13, fontweight = 'bold')
ax1.grid(True, alpha = 0.3)
cbar1 = plt.colorbar(ax1.collections[0], ax = ax1)
cbar1.set_label('Chunk ID', fontsize = 10)

# Bottom: Chunk distribution and statistics
ax2 = axes[1]
chunk_counts = chunked_df['chunk_id'].value_counts().sort_index()
chunk_stats = chunked_df.groupby('chunk_id').agg({
    'glucose': ['count', 'mean', 'std'],
    'N0': 'mean' if 'N0' in chunked_df.columns else 'count'
}).reset_index()

# Plot chunk sizes
ax2_twin = ax2.twinx()
bars = ax2.bar(chunk_counts.index[:50], chunk_counts.values[:50], 
              alpha = 0.7, color = 'steelblue', edgecolor = 'black', linewidth = 0.5)
ax2.set_xlabel('Chunk ID', fontsize = 11)
ax2.set_ylabel('Number of Samples per Chunk', fontsize = 11, color = 'steelblue')
ax2.tick_params(axis = 'y', labelcolor = 'steelblue')
ax2.set_title('Chunk Statistics: Sample Distribution (First 50 Chunks)', 
             fontsize = 13, fontweight = 'bold')
ax2.grid(True, alpha = 0.3, axis = 'y')

# Plot mean glucose per chunk
if 'glucose' in chunk_stats.columns:
    line = ax2_twin.plot(chunk_stats['chunk_id'][:50], 
                        chunk_stats[('glucose', 'mean')][:50], 
                        color = 'red', linewidth = 2, marker = 'o', markersize = 4, 
                        label = 'Mean Glucose')
    ax2_twin.set_ylabel('Mean Glucose (mg/dL)', fontsize = 11, color = 'red')
    ax2_twin.tick_params(axis = 'y', labelcolor = 'red')
    ax2_twin.legend(loc = 'upper right')

plt.tight_layout()
plt.savefig('chunk_splitting.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: chunk_splitting.png")
print(f"Total chunks: {chunked_df['chunk_id'].nunique()}")
print(f"Average samples per chunk: {chunked_df.groupby('chunk_id').size().mean():.1f}")


### 6.3 Train Models and Generate Model Plots


In [ ]:
# Prepare data for model training
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import RobustScaler

# Split data
chunked_df = mdl.create_time_chunks(features_df, chunk_hours = 1)
splitter = GroupShuffleSplit(n_splits = 1, train_size = 0.8, random_state = 42)
train_idx, test_idx = next(splitter.split(chunked_df, groups = chunked_df['chunk_id']))
df_train, df_test = chunked_df.iloc[train_idx], chunked_df.iloc[test_idx]

# Prepare features
numeric_train = df_train.select_dtypes(include = [np.number])
numeric_test = df_test.select_dtypes(include = [np.number])
metadata = ['glucose', 'chunk_id', 'hours_elapsed', 'session_id', 'DATE']
feature_cols_model = [c for c in numeric_train.columns if c not in metadata]

X_train = numeric_train[feature_cols_model].fillna(0).replace([np.inf, -np.inf], 0)
X_test = numeric_test[feature_cols_model].fillna(0).replace([np.inf, -np.inf], 0)
y_train = numeric_train['glucose']
y_test = numeric_test['glucose']

# Scale features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Features: {len(feature_cols_model)}")


### 6.4 Gradient Boosting Model Visualization


In [ ]:
# Train GB model
print("Training Gradient Boosting model...")
gb_model = HistGradientBoostingRegressor(max_iter = 1000, learning_rate = 0.05, 
                                         max_depth = 8, l2_regularization = 5.0, 
                                         random_state = 42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)

# Calculate metrics
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)
gb_mard = mdl.calculate_mard(y_test, gb_pred)

print(f"GB Model - MAE: {gb_mae:.2f}, R²: {gb_r2:.4f}, MARD: {gb_mard:.2f}%")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize = (16, 6))

# Left: Prediction vs Actual scatter
ax1 = axes[0]
sample_size = min(2000, len(y_test))
sample_idx = np.random.choice(len(y_test), sample_size, replace = False)
y_sample = y_test.iloc[sample_idx].values
pred_sample = gb_pred[sample_idx]

ax1.scatter(y_sample, pred_sample, alpha = 0.5, s = 20, c = 'steelblue', edgecolors = 'black', linewidth = 0.3)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
        'r--', lw = 2, label = 'Perfect Prediction')
ax1.set_xlabel('Actual Glucose (mg/dL)', fontsize = 12, fontweight = 'bold')
ax1.set_ylabel('Predicted Glucose (mg/dL)', fontsize = 12, fontweight = 'bold')
ax1.set_title(f'Gradient Boosting: Predictions vs Actual\nMAE: {gb_mae:.2f} mg/dL, R²: {gb_r2:.4f}, MARD: {gb_mard:.2f}%', 
             fontsize = 13, fontweight = 'bold')
ax1.legend()
ax1.grid(True, alpha = 0.3)

# Right: Residual plot
ax2 = axes[1]
residuals = gb_pred - y_test.values
ax2.scatter(y_sample, residuals[sample_idx], alpha = 0.5, s = 20, c = 'coral', edgecolors = 'black', linewidth = 0.3)
ax2.axhline(y = 0, color = 'black', linestyle = '--', lw = 2)
ax2.axhline(y = 20, color = 'red', linestyle = ':', lw = 1.5, label = '±20 mg/dL')
ax2.axhline(y = -20, color = 'red', linestyle = ':', lw = 1.5)
ax2.set_xlabel('Actual Glucose (mg/dL)', fontsize = 12, fontweight = 'bold')
ax2.set_ylabel('Residual (Predicted - Actual) (mg/dL)', fontsize = 12, fontweight = 'bold')
ax2.set_title('Gradient Boosting: Residual Analysis', fontsize = 13, fontweight = 'bold')
ax2.legend()
ax2.grid(True, alpha = 0.3)

plt.tight_layout()
plt.savefig('gb_model.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: gb_model.png")


### 6.5 Extra Trees Model Visualization


In [ ]:
# Train ET model
print("Training Extra Trees model...")
et_model = ExtraTreesRegressor(n_estimators = 50, min_samples_leaf = 50, 
                               n_jobs = 1, random_state = 42)
et_model.fit(X_train_scaled, y_train)
et_pred = et_model.predict(X_test_scaled)

# Calculate metrics
et_mae = mean_absolute_error(y_test, et_pred)
et_r2 = r2_score(y_test, et_pred)
et_mard = mdl.calculate_mard(y_test, et_pred)

print(f"ET Model - MAE: {et_mae:.2f}, R²: {et_r2:.4f}, MARD: {et_mard:.2f}%")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize = (16, 6))

# Left: Prediction vs Actual scatter
ax1 = axes[0]
sample_size = min(2000, len(y_test))
sample_idx = np.random.choice(len(y_test), sample_size, replace = False)
y_sample = y_test.iloc[sample_idx].values
pred_sample = et_pred[sample_idx]

ax1.scatter(y_sample, pred_sample, alpha = 0.5, s = 20, c = '#A23B72', edgecolors = 'black', linewidth = 0.3)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
        'r--', lw = 2, label = 'Perfect Prediction')
ax1.set_xlabel('Actual Glucose (mg/dL)', fontsize = 12, fontweight = 'bold')
ax1.set_ylabel('Predicted Glucose (mg/dL)', fontsize = 12, fontweight = 'bold')
ax1.set_title(f'Extra Trees: Predictions vs Actual\nMAE: {et_mae:.2f} mg/dL, R²: {et_r2:.4f}, MARD: {et_mard:.2f}%', 
             fontsize = 13, fontweight = 'bold')
ax1.legend()
ax1.grid(True, alpha = 0.3)

# Right: Residual plot
ax2 = axes[1]
residuals = et_pred - y_test.values
ax2.scatter(y_sample, residuals[sample_idx], alpha = 0.5, s = 20, c = '#F18F01', edgecolors = 'black', linewidth = 0.3)
ax2.axhline(y = 0, color = 'black', linestyle = '--', lw = 2)
ax2.axhline(y = 20, color = 'red', linestyle = ':', lw = 1.5, label = '±20 mg/dL')
ax2.axhline(y = -20, color = 'red', linestyle = ':', lw = 1.5)
ax2.set_xlabel('Actual Glucose (mg/dL)', fontsize = 12, fontweight = 'bold')
ax2.set_ylabel('Residual (Predicted - Actual) (mg/dL)', fontsize = 12, fontweight = 'bold')
ax2.set_title('Extra Trees: Residual Analysis', fontsize = 13, fontweight = 'bold')
ax2.legend()
ax2.grid(True, alpha = 0.3)

plt.tight_layout()
plt.savefig('et_model.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: et_model.png")


### 6.6 Overall Feature Importance (Combined GB + ET)


In [ ]:
# Get feature importances from both models
gb_importances = gb_model.feature_importances_
et_importances = et_model.feature_importances_

# Normalize importances
gb_norm = (gb_importances - gb_importances.min()) / (gb_importances.max() - gb_importances.min() + 1e-10)
et_norm = (et_importances - et_importances.min()) / (et_importances.max() - et_importances.min() + 1e-10)

# Combined importance (average of normalized)
combined_importance = (gb_norm + et_norm) / 2

# Create DataFrame
importance_df = pd.DataFrame({
    'feature': feature_cols_model,
    'gb_importance': gb_importances,
    'et_importance': et_importances,
    'combined_importance': combined_importance
}).sort_values('combined_importance', ascending = False)

# Get top 25 features
top_25 = importance_df.head(25)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize = (18, 10))

# Left: Combined importance
ax1 = axes[0]
top_combined = top_25.sort_values('combined_importance', ascending = True)
feature_names_clean = [name.replace('_', ' ').title() for name in top_combined['feature']]
feature_names_clean = [name.replace('Normalized', 'Norm').replace('Rolling', 'Roll')
                      .replace('Gradient', 'Grad').replace('Percentage', 'Pct')
                      .replace('Temperature', 'Temp') for name in feature_names_clean]

y_pos = np.arange(len(feature_names_clean))
bars1 = ax1.barh(y_pos, top_combined['combined_importance'].values, 
                color = '#2E86AB', alpha = 0.8, edgecolor = 'black', linewidth = 0.5)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(feature_names_clean, fontsize = 10)
ax1.set_xlabel('Combined Normalized Importance', fontsize = 12, fontweight = 'bold')
ax1.set_title('Top 25 Features: Combined Importance\n(GB + ET Average)', 
             fontsize = 14, fontweight = 'bold')
ax1.grid(True, alpha = 0.3, axis = 'x')

# Right: Comparison of GB vs ET importance
ax2 = axes[1]
x_pos = np.arange(len(feature_names_clean))
width = 0.35

bars2a = ax2.barh(y_pos - width/2, top_combined['gb_importance'].values, 
                  width, label = 'Gradient Boosting', color = '#2E86AB', alpha = 0.7, 
                  edgecolor = 'black', linewidth = 0.5)
bars2b = ax2.barh(y_pos + width/2, top_combined['et_importance'].values, 
                  width, label = 'Extra Trees', color = '#A23B72', alpha = 0.7, 
                  edgecolor = 'black', linewidth = 0.5)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(feature_names_clean, fontsize = 10)
ax2.set_xlabel('Feature Importance', fontsize = 12, fontweight = 'bold')
ax2.set_title('Top 25 Features: GB vs ET Importance Comparison', 
             fontsize = 14, fontweight = 'bold')
ax2.legend(fontsize = 11)
ax2.grid(True, alpha = 0.3, axis = 'x')

plt.tight_layout()
plt.savefig('overall_feature_importance.png', dpi = 300, bbox_inches = 'tight')
plt.show()
print("Saved: overall_feature_importance.png")


## 7. Final Summary

All visualization files have been generated:
- `feature_engineering_overview.png` - Feature count and categories
- `feature_correlations.png` - Top features by correlation
- `feature_importance_analysis.png` - F-score and combined importance
- `feature_examples.png` - Before/after feature engineering examples
- `feature_statistics.png` - Feature statistics by category
- `signal_drift_sessions.png` - Signal drift Session A vs Session B
- `chunk_splitting.png` - 1-hour chunk splitting visualization
- `gb_model.png` - Gradient Boosting model performance
- `et_model.png` - Extra Trees model performance
- `overall_feature_importance.png` - Combined feature importance (GB + ET)


## 5. Summary

All visualization files have been saved:
- `feature_engineering_overview.png` - Feature count and categories
- `feature_correlations.png` - Top features by correlation
- `feature_importance_analysis.png` - F-score and combined importance
- `feature_examples.png` - Before/after feature engineering examples
- `feature_statistics.png` - Feature statistics by category
